# Perfilado del dataset · COVID Tracking Project

**Objetivo:** entender qué tiene el dataset antes de analizarlo: forma, tipos,
nulos, duplicados, rangos y calidad declarada.

**Fuentes:** `src/eda/load.py` (ingesta) y `src.eda.clean` (limpieza básica).
El notebook **no duplica** lógica: solo llama y muestra.


## 1. Configuración e imports


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.eda.clean import audit_cleaning, build_clean_dataset
from src.eda.load import load_covid_data

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)


## 2. Carga del dataset crudo

`load_covid_data()` descarga el CSV si no hay copia local y valida el esquema
(≥ 1000 filas, ≥ 8 columnas y columnas obligatorias).


In [ ]:
raw = load_covid_data()
print("shape:", raw.shape)
raw.head(3)


## 3. Forma y tipos

El dataset tiene una fila por `state` y `date`. `date` llega como entero
`YYYYMMDD` y las columnas de conteo como `float64` por la presencia de
nulos.


In [ ]:
print("filas:", len(raw))
print("columnas:", raw.shape[1])
print("estados:", raw["state"].nunique())
print("rango de fechas:", raw["date"].min(), "->", raw["date"].max())
print()
raw.info()


## 4. Valores nulos

Se calcula el porcentaje de nulos por columna y se ordena de mayor a menor.
Las columnas con más del 80 % se excluyen del análisis principal.


In [ ]:
missing = (
    raw.isna().mean().mul(100).round(1).sort_values(ascending=False)
)
print("Columnas con algún nulo:", int((missing > 0).sum()))
print("Columnas con > 80 % de nulos:", int((missing > 80).sum()))
missing.head(20).to_frame("pct_nulos")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
top = missing.head(15).sort_values()
top.plot.barh(ax=ax, color="#4C78A8")
ax.set_title("15 columnas con mayor porcentaje de nulos")
ax.set_xlabel("Porcentaje de nulos")
ax.set_ylabel("")
for i, v in enumerate(top):
    ax.text(v + 0.5, i, f"{v:.0f}%", va="center", fontsize=8)
fig.tight_layout()
plt.show()


## 5. Duplicados por `(state, date)`

La clave natural del dataset es la pareja estado–fecha. Si hubiera duplicados,
la limpieza conserva el registro con `dateModified` más reciente.


In [ ]:
dupes = raw.duplicated(subset=["state", "date"]).sum()
print("duplicados (state, date):", int(dupes))
print("estados:", sorted(raw["state"].unique().tolist()))


## 6. Estadísticos básicos de las variables clave


In [ ]:
raw[["positive", "death", "totalTestResults", "hospitalizedCurrently"]].describe().T


## 7. Incrementos negativos

Los incrementos diarios pueden ser negativos por correcciones de reporting.
No se borran: se cuentan para decidir el tratamiento en la limpieza avanzada.


In [ ]:
increment_cols = [
    "positiveIncrease",
    "negativeIncrease",
    "totalTestResultsIncrease",
    "deathIncrease",
    "hospitalizedIncrease",
]
neg_counts = {c: int((raw[c] < 0).sum()) for c in increment_cols if c in raw.columns}
pd.Series(neg_counts, name="incrementos_negativos")


## 8. Limpieza básica y auditoría

`build_clean_dataset()` carga, limpia (fechas + duplicados), guarda el CSV
procesado y devuelve la auditoría. `data/raw/` no se toca.


In [ ]:
clean, audit = build_clean_dataset()
print("auditoría:", audit)
print("shape limpio:", clean.shape)
print("date dtype:", clean["date"].dtype)
clean.head(3)


## 9. Comparativa antes / después

| Aspecto | Crudo | Limpio |
|---|---|---|
| Filas | 20780 | 20780 |
| Fechas inválidas | 0 | 0 |
| Duplicados | 0 | 0 |
| Rango | 2020-01-13 → 2021-03-07 | 2020-01-13 → 2021-03-07 |

La limpieza básica no elimina filas en esta copia (0 fechas inválidas y 0
duplicados), pero el código y los tests cubren ambos casos para otras
descargas.


## 10. Columnas con más del 80 % de nulos

Se excluyen del análisis principal en la limpieza avanzada. La lista completa
está en `docs/data_dictionary.md`.


In [ ]:
missing_all = raw.isna().mean().mul(100).round(1)
high_null = missing_all[missing_all > 80].sort_values(ascending=False)
print("columnas excluidas (> 80 % nulos):", len(high_null))
high_null.to_frame("pct_nulos")


## 11. Conclusiones del perfilado

- El dataset tiene **20 780 filas** y **56 columnas**, con cobertura de 56
  códigos (50 estados + DC + territorios).
- El periodo va del 2020-01-13 al 2021-03-07 (última foto publicada).
- `date` llega como entero `YYYYMMDD` y se convierte a `datetime64[ns]`.
- No hay duplicados por `(state, date)` ni fechas inválidas en esta copia.
- Hay **15 columnas con más del 80 % de nulos** (`dataQualityGrade` y
  `grade` al 100 %), que se excluirán del análisis.
- Los 5 incrementos diarios tienen valores negativos (24 a 88 según la
  columna), que se tratan en la limpieza avanzada.
- `totalTestResults` no es homogéneo en el tiempo ni entre estados.

**Siguiente paso:** `N1-05` define las preguntas de negocio y las hipótesis
antes de cualquier análisis.
